# 00 — Sanity Checks

Minimal working pipeline validation:
1. Generate a hypergraph
2. Compute ⟨Z_a⟩_{q_θ} (single observable)
3. Compare MC estimator vs exact for n ≤ 10
4. Compute one gradient estimate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from iqp_bp.hypergraph.families import bounded_degree
from iqp_bp.iqp.expectation import iqp_expectation, iqp_expectation_exact
from iqp_bp.mmd.loss import mmd2
from iqp_bp.mmd.gradients import grad_mmd2_analytic, grad_mmd2_finite_diff

rng = np.random.default_rng(42)
print('Imports OK')

## 1. Generate hypergraph

In [ ]:
n = 8
m = n
G = bounded_degree(n=n, m=m, max_weight=3, rng=np.random.default_rng(0))
theta = rng.uniform(-np.pi, np.pi, size=m)
a = rng.integers(0, 2, size=n, dtype=np.uint8)
a[0] = 1  # ensure nonzero

print(f'G shape: {G.shape}')
print(f'Generator weights: {G.sum(axis=1)}')
print(f'Observable a: {a}')

## 2. ⟨Z_a⟩ estimator vs exact

In [ ]:
exact = iqp_expectation_exact(theta, G, a)

estimates = []
for nz in [64, 256, 1024, 4096]:
    est, se = iqp_expectation(theta, G, a, num_z_samples=nz, rng=np.random.default_rng(1))
    estimates.append((nz, est, se))
    print(f'nz={nz:5d}: est={est:.4f} ± {se:.4f}  |  exact={exact:.4f}  |  err={abs(est-exact):.4f}')

## 3. MMD² estimate

In [ ]:
data = rng.integers(0, 2, size=(1000, n), dtype=np.uint8)
loss = mmd2(theta, G, data, kernel='gaussian', sigma=1.0,
            num_a_samples=256, num_z_samples=512, rng=np.random.default_rng(2))
print(f'MMD² estimate: {loss:.6f}')

## 4. Gradient estimate

In [ ]:
grad_a = grad_mmd2_analytic(theta, G, data, param_idx=0,
                             kernel='gaussian', sigma=1.0,
                             num_a_samples=256, num_z_samples=512,
                             rng=np.random.default_rng(3))
grad_fd = grad_mmd2_finite_diff(theta, G, data, param_idx=0,
                                 kernel='gaussian', sigma=1.0,
                                 num_a_samples=256, num_z_samples=1024,
                                 rng=np.random.default_rng(4))
print(f'Analytic gradient:       {grad_a:.6f}')
print(f'Finite-diff gradient:    {grad_fd:.6f}')
print(f'Relative difference:     {abs(grad_a - grad_fd):.6f}')